# ML Forecasting of Monthly Wind Speed in Bangladesh (1982-2024)

**Ashiqur Rahman Khan** | BSc Statistics, MBSTU | Supervisor: Somaresh Kumar Mondal

Benchmarks Random Forest, XGBoost, SVR, LSTM and a Hybrid SARIMA+XGBoost against the
SARIMA(0,1,1)(0,1,1)[12] baseline produced in R.

**Design (must match the R pipeline exactly)**

| Item | Value |
|---|---|
| Feature runway | 1981-01 to 1981-12 (12 months, inputs only) |
| Train | 1982-01 to 2022-12 (492 months) |
| Holdout | 2023-01 to 2024-12 (24 months) |
| CV | Rolling-origin, 5 folds x 24-month windows, expanding |
| Features | 21 predictors |

**Leakage discipline.** A leakage bug was found and fixed in June 2026: rolling features had
been computed with `align="right"`, which includes the current month's actual wind speed -
the target being predicted. Every rolling feature here is therefore built as
`.shift(1).rolling(w)`, and every scaler is fit on the training slice only.
Section 4 runs an automated audit that fails loudly if either rule is broken.

## 1. Setup

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import os, glob

# ---- Locate the project folder --------------------------------------------
MYDRIVE = '/content/drive/MyDrive'
assert os.path.isdir(MYDRIVE), 'Drive not mounted - run the cell above first'

cands = [p for p in glob.glob(os.path.join(MYDRIVE, '*'))
         if os.path.isdir(p) and 'paper publication' in os.path.basename(p).lower()]
assert cands, 'Paper Publication folder not found in MyDrive'
ROOT = cands[0]

# ---- Find ML_Input wherever it lives under ROOT ----------------------------
hits = [d for d, subdirs, files in os.walk(ROOT) if os.path.basename(d) == 'ML_Input']

if not hits:
    print('ML_Input not found. Directory tree under ROOT:')
    for d, subdirs, files in os.walk(ROOT):
        depth = d.replace(ROOT, '').count(os.sep)
        if depth <= 2:
            print('  ' * depth, f'[{os.path.basename(d) or "ROOT"}]', f'({len(files)} files)')
    raise FileNotFoundError('Set ML_IN manually from the tree above')

ML_IN = hits[0]

# ---- Outputs: sibling of SARIMA_Output, at project root -------------------
ML_OUT    = os.path.join(ROOT, 'ML_Output')
OUT_DATA  = os.path.join(ML_OUT, 'Data')
OUT_GRAPH = os.path.join(ML_OUT, 'Graph')
OUT_CKPT  = os.path.join(ML_OUT, 'Checkpoints')

for d in (ML_OUT, OUT_DATA, OUT_GRAPH, OUT_CKPT):
    os.makedirs(d, exist_ok=True)

print(f'ROOT  : [{os.path.basename(ROOT)}]')
print('Input :', ML_IN.replace(ROOT, '...'))
print('Data  :', OUT_DATA.replace(ROOT, '...'))
print('Graph :', OUT_GRAPH.replace(ROOT, '...'))
print('Ckpt  :', OUT_CKPT.replace(ROOT, '...'))
print()

need = ['00b_analysis_series_1981_2024.csv',
        '00c_exogenous_variables_1981_2024.csv',
        '08b_holdout_forecast_sarima.csv',
        '10c_cv_forecasts_sarima.csv']
have = sorted(os.listdir(ML_IN))
print('Files in ML_Input:')
for f in have:
    print('  ', f)

missing = [f for f in need if f not in have]
assert not missing, f'missing required files: {missing}'
print()
print('All 4 required inputs present.')

ROOT  : [Paper Publication folder ]
Input : .../SARIMA_Output/ML_Input
Data  : .../ML_Output/Data
Graph : .../ML_Output/Graph
Ckpt  : .../ML_Output/Checkpoints

Files in ML_Input:
   00b_analysis_series_1981_2024.csv
   00c_exogenous_variables_1981_2024.csv
   08b_holdout_forecast_sarima.csv
   10c_cv_forecasts_sarima.csv
   MANIFEST.csv

All 4 required inputs present.


In [24]:
import random, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import xgboost as xgb
import statsmodels.api as sm

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

warnings.filterwarnings('ignore')

SEED = 42
def set_seeds(s=SEED):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s)
    os.environ['PYTHONHASHSEED'] = str(s)
set_seeds()

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
print('numpy', np.__version__, '| pandas', pd.__version__,
      '| xgboost', xgb.__version__, '| tensorflow', tf.__version__)

numpy 2.1.3 | pandas 2.2.3 | xgboost 3.4.1 | tensorflow 2.20.0


## 2. Load inputs

Both files run 1981-01 to 2024-12 (528 rows). The 12 rows of 1981 exist only to seed the
lag-12 and rolling-12 features; they are never trained on and never predicted.

In [25]:
ws  = pd.read_csv(os.path.join(ML_IN, '00b_analysis_series_1981_2024.csv'),
                  parse_dates=['Date'])
exo = pd.read_csv(os.path.join(ML_IN, '00c_exogenous_variables_1981_2024.csv'),
                  parse_dates=['Date'])

print('target series :', ws.shape, ws.Date.min().date(), '->', ws.Date.max().date())
print('exogenous     :', exo.shape, exo.Date.min().date(), '->', exo.Date.max().date())
print()
print(ws.Role.value_counts().reindex(['Feature_runway','Train','Holdout']))

# Merge on Date; keep only the exogenous columns actually used
EXO_COLS = ['PS', 'T2M', 'RH2M', 'WD2M', 'Nino34_Anomaly', 'IOD_DMI']
df = ws.merge(exo[['Date'] + EXO_COLS], on='Date', how='left')

assert len(df) == 528, f'expected 528 rows, got {len(df)}'
assert df[EXO_COLS].isna().sum().sum() == 0, 'NA in exogenous columns after merge'
assert (df.Date.diff().dt.days.dropna() > 0).all(), 'dates not sorted / duplicated'
print(); print('Merged:', df.shape)
df.head(3)

target series : (528, 3) 1981-01-01 -> 2024-12-01
exogenous     : (528, 14) 1981-01-01 -> 2024-12-01

Role
Feature_runway     12
Train             492
Holdout            24
Name: count, dtype: int64

Merged: (528, 9)


,Date,WS,Role,PS,T2M,RH2M,WD2M,Nino34_Anomaly,IOD_DMI
0,1981-01-01,2.253589,Feature_runway,100.49,18.87,65.04,233.23,-0.32,-0.201
1,1981-02-01,2.386490,Feature_runway,100.24,20.97,59.31,250.73,-0.39,-0.024
2,1981-03-01,2.589980,Feature_runway,100.08,25.01,59.40,221.51,0.14,0.027


## 3. Feature engineering

21 predictors, matching the locked specification.

| Group | Features |
|---|---|
| AR lags | `WS_lag1, WS_lag2, WS_lag3, WS_lag6, WS_lag12` |
| Rolling | `WS_roll3, WS_roll12, WS_std3` |
| Calendar | `Month` |
| Meteorological | `PS, T2M, RH2M, WD2M` |
| Teleconnections | `Nino34_Anomaly, IOD_DMI` |
| ENSO lagged | `Nino34_lag2, Nino34_lag3, Nino34_lag6` |
| IOD lagged | `IOD_lag2, IOD_lag3, IOD_lag6` |

**The rolling features are the ones that carried the original bug.** `.shift(1)` moves the window back one month *before* aggregating, so `WS_roll3` at month *t* averages
*t-3, t-2, t-1* and never touches *t* itself. Writing `.rolling(3).mean()` without the shift
is the leaking version.

In [26]:
d = df.copy()

# --- AR lags ---------------------------------------------------------------
for L in [1, 2, 3, 6, 12]:
    d[f'WS_lag{L}'] = d['WS'].shift(L)

# --- Rolling statistics: shift(1) FIRST, then aggregate --------------------
d['WS_roll3']  = d['WS'].shift(1).rolling(3).mean()
d['WS_roll12'] = d['WS'].shift(1).rolling(12).mean()
d['WS_std3']   = d['WS'].shift(1).rolling(3).std()

# --- Calendar --------------------------------------------------------------
d['Month'] = d['Date'].dt.month

# --- Teleconnection lags ---------------------------------------------------
for L in [2, 3, 6]:
    d[f'Nino34_lag{L}'] = d['Nino34_Anomaly'].shift(L)
    d[f'IOD_lag{L}']    = d['IOD_DMI'].shift(L)

FEATURES = [
    'WS_lag1','WS_lag2','WS_lag3','WS_lag6','WS_lag12',
    'WS_roll3','WS_roll12','WS_std3',
    'Month',
    'PS','T2M','RH2M','WD2M',
    'Nino34_Anomaly','IOD_DMI',
    'Nino34_lag2','Nino34_lag3','Nino34_lag6',
    'IOD_lag2','IOD_lag3','IOD_lag6',
]
TARGET = 'WS'
assert len(FEATURES) == 21, f'expected 21 features, got {len(FEATURES)}'

# --- Drop the runway rows: they were inputs, never targets -----------------
mat = d[d['Role'] != 'Feature_runway'].reset_index(drop=True)

print('rows after dropping runway :', len(mat))
print('date range                 :', mat.Date.min().date(), '->', mat.Date.max().date())
print('NA in feature matrix       :', int(mat[FEATURES].isna().sum().sum()))

assert len(mat) == 516, f'expected 516 rows, got {len(mat)}'
assert mat[FEATURES].isna().sum().sum() == 0, 'NA remain - runway was too short'
assert mat.Date.min() == pd.Timestamp('1982-01-01')
assert mat.Date.max() == pd.Timestamp('2024-12-01')
print(); print('Feature matrix OK: 516 x', len(FEATURES))

rows after dropping runway : 516
date range                 : 1982-01-01 -> 2024-12-01
NA in feature matrix       : 0

Feature matrix OK: 516 x 21


## 4. Leakage audit

Three checks that fail loudly rather than silently producing inflated scores.

1. **Rolling correctness** - recompute each rolling feature independently and compare
   against both the correct (shifted) and the buggy (unshifted) version.
2. **Target correlation** - a rolling feature that leaks correlates with `WS` at almost 1.0.
3. **Row alignment** - the holdout block is exactly the 24 months the R pipeline used.

In [27]:
audit = []

# --- 1. rolling features must equal the SHIFTED version, not the unshifted one
checks = {
    'WS_roll3':  (d['WS'].shift(1).rolling(3).mean(),  d['WS'].rolling(3).mean()),
    'WS_roll12': (d['WS'].shift(1).rolling(12).mean(), d['WS'].rolling(12).mean()),
    'WS_std3':   (d['WS'].shift(1).rolling(3).std(),   d['WS'].rolling(3).std()),
}
for name, (correct, buggy) in checks.items():
    ok_correct = np.allclose(d[name].dropna(), correct.dropna(), atol=1e-12)
    matches_buggy = np.allclose(d[name].dropna().values[-100:],
                                buggy.dropna().values[-100:], atol=1e-12)
    audit.append((f'{name} == shifted version', ok_correct))
    audit.append((f'{name} != unshifted (buggy) version', not matches_buggy))

# --- 2. correlation with the target (leaking rollers sit at ~0.99)
for name in ['WS_roll3', 'WS_roll12', 'WS_std3']:
    r = mat[name].corr(mat[TARGET])
    audit.append((f'corr({name}, WS) = {r:+.3f} below 0.97', abs(r) < 0.97))

# --- 3. split alignment
hold_dates = mat.loc[mat.Role == 'Holdout', 'Date']
audit.append(('holdout = 24 months', len(hold_dates) == 24))
audit.append(('holdout starts 2023-01', hold_dates.min() == pd.Timestamp('2023-01-01')))
audit.append(('holdout ends 2024-12',  hold_dates.max() == pd.Timestamp('2024-12-01')))
audit.append(('train = 492 months', int((mat.Role == 'Train').sum()) == 492))

print('LEAKAGE AUDIT')
print('-' * 58)
for label, passed in audit:
    print(f'  [{"PASS" if passed else "FAIL"}]  {label}')
print('-' * 58)

failed = [l for l, p in audit if not p]
if failed:
    raise AssertionError('Leakage audit FAILED: ' + '; '.join(failed))
print('All checks passed.')

LEAKAGE AUDIT
----------------------------------------------------------
  [PASS]  WS_roll3 == shifted version
  [PASS]  WS_roll3 != unshifted (buggy) version
  [PASS]  WS_roll12 == shifted version
  [PASS]  WS_roll12 != unshifted (buggy) version
  [PASS]  WS_std3 == shifted version
  [PASS]  WS_std3 != unshifted (buggy) version
  [PASS]  corr(WS_roll3, WS) = +0.444 below 0.97
  [PASS]  corr(WS_roll12, WS) = +0.199 below 0.97
  [PASS]  corr(WS_std3, WS) = -0.014 below 0.97
  [PASS]  holdout = 24 months
  [PASS]  holdout starts 2023-01
  [PASS]  holdout ends 2024-12
  [PASS]  train = 492 months
----------------------------------------------------------
All checks passed.


## 5. Split and shared helpers

`fit_scalers` is deliberately the only place scaling happens, and it only ever sees
training rows. Everything downstream calls it, so there is one place to audit rather
than five.

In [28]:
TRAIN_END  = pd.Timestamp('2022-12-01')
TEST_START = pd.Timestamp('2023-01-01')

tr_idx = mat.index[mat.Date <= TRAIN_END].to_numpy()
te_idx = mat.index[mat.Date >= TEST_START].to_numpy()

X_all = mat[FEATURES].to_numpy(dtype=float)
y_all = mat[TARGET].to_numpy(dtype=float)
dates_all = mat['Date'].to_numpy()

print(f'train {len(tr_idx)} rows | holdout {len(te_idx)} rows')


def metrics(actual, pred, model):
    actual = np.asarray(actual, float); pred = np.asarray(pred, float)
    e = actual - pred
    return {
        'Model': model,
        'RMSE': round(float(np.sqrt(np.mean(e ** 2))), 4),
        'MAE' : round(float(np.mean(np.abs(e))), 4),
        'MAPE': round(float(100 * np.mean(np.abs(e / actual))), 4),
        'R2'  : round(float(1 - np.sum(e ** 2) / np.sum((actual - actual.mean()) ** 2)), 4),
        'n'   : int(len(actual)),
    }


def fit_scalers(X, y, train_pos):
    """Fit on TRAIN ROWS ONLY, then transform everything. Single choke point."""
    xs = StandardScaler().fit(X[train_pos])
    ys = StandardScaler().fit(y[train_pos].reshape(-1, 1))
    return xs, ys, xs.transform(X), ys.transform(y.reshape(-1, 1)).ravel()


def save_ckpt(name, obj):
    p = os.path.join(OUT_CKPT, f'{name}.json')
    with open(p, 'w') as f:
        json.dump(obj, f, indent=2, default=str)
    print(f'  checkpoint -> Checkpoints/{name}.json')

train 492 rows | holdout 24 rows


## 6. SARIMA in Python: cross-check and hybrid base

The canonical SARIMA numbers come from R (`forecast::Arima`). Refitting the same order
here serves two purposes: it validates that both languages agree, and the hybrid model
needs per-fold residuals that the R export does not contain.

If the Python holdout RMSE lands near R's **0.2469**, the two implementations agree and the
hybrid rests on a sound base.

In [29]:
SARIMA_ORDER    = (0, 1, 1)
SARIMA_SEASONAL = (0, 1, 1, 12)


def fit_sarima(y_train, h):
    """Fit SARIMA on y_train, return (in-sample fitted, h-step forecast)."""
    m = sm.tsa.SARIMAX(y_train, order=SARIMA_ORDER, seasonal_order=SARIMA_SEASONAL,
                       enforce_stationarity=False, enforce_invertibility=False)
    r = m.fit(disp=False)
    return np.asarray(r.fittedvalues, float), np.asarray(r.forecast(steps=h), float)


sarima_fitted_tr, sarima_fc_te = fit_sarima(y_all[tr_idx], len(te_idx))
py_sarima = metrics(y_all[te_idx], sarima_fc_te, 'SARIMA (python check)')

# R's canonical holdout numbers
r_sarima = pd.read_csv(os.path.join(ML_IN, '08b_holdout_forecast_sarima.csv'),
                       parse_dates=['Date'])
r_rmse = float(np.sqrt(np.mean(r_sarima['Error'].to_numpy() ** 2)))

print(f"Python SARIMA holdout RMSE : {py_sarima['RMSE']:.4f}")
print(f'R      SARIMA holdout RMSE : {r_rmse:.4f}')
print(f'Absolute difference        : {abs(py_sarima["RMSE"] - r_rmse):.4f}')
if abs(py_sarima['RMSE'] - r_rmse) > 0.02:
    print()
    print('WARNING: implementations differ by more than 0.02 -'
          ' investigate before trusting the hybrid.')
else:
    print()
    print('Implementations agree. R remains canonical for reported SARIMA numbers.')

Python SARIMA holdout RMSE : 0.2464
R      SARIMA holdout RMSE : 0.2469
Absolute difference        : 0.0005

Implementations agree. R remains canonical for reported SARIMA numbers.


## 7. Model definitions

Each returns predictions for the supplied test rows. All five take the same
`(X, y, train_pos, test_pos)` signature so the holdout run and the CV loop share code.

Hyperparameters are modest on purpose: with 492 training months, heavy tuning would
overfit and would also need a nested inner-CV loop to stay honest.

In [30]:
def run_rf(X, y, tr_pos, te_pos):
    set_seeds()
    m = RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=2,
                              max_features='sqrt', random_state=SEED, n_jobs=-1)
    m.fit(X[tr_pos], y[tr_pos])
    return m.predict(X[te_pos]), m


def run_xgb(X, y, tr_pos, te_pos):
    set_seeds()
    m = xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         reg_lambda=1.0, random_state=SEED, n_jobs=-1,
                         objective='reg:squarederror')
    m.fit(X[tr_pos], y[tr_pos], verbose=False)
    return m.predict(X[te_pos]), m


def run_svr(X, y, tr_pos, te_pos):
    set_seeds()
    xs, ys, Xs, yscaled = fit_scalers(X, y, tr_pos)      # train-only fit
    m = SVR(kernel='rbf', C=10.0, gamma='scale', epsilon=0.05)
    m.fit(Xs[tr_pos], yscaled[tr_pos])
    pred_s = m.predict(Xs[te_pos])
    return ys.inverse_transform(pred_s.reshape(-1, 1)).ravel(), m


SEQ_LEN = 12

def make_sequences(Xs, yv, positions, seq_len=SEQ_LEN):
    """Sequence ending at position i uses rows i-seq_len+1 .. i (history only)."""
    Xo, yo, keep = [], [], []
    for i in positions:
        if i - seq_len + 1 < 0:
            continue
        Xo.append(Xs[i - seq_len + 1: i + 1])
        yo.append(yv[i])
        keep.append(i)
    return np.asarray(Xo), np.asarray(yo), np.asarray(keep)


def run_lstm(X, y, tr_pos, te_pos, epochs=200, verbose=0):
    set_seeds()
    xs, ys, Xs, yscaled = fit_scalers(X, y, tr_pos)      # train-only fit

    Xtr, ytr, _ = make_sequences(Xs, yscaled, tr_pos)
    Xte, _, kept = make_sequences(Xs, yscaled, te_pos)

    model = Sequential([
        Input(shape=(SEQ_LEN, X.shape[1])),
        LSTM(64, return_sequences=True), Dropout(0.2),
        LSTM(32), Dropout(0.2),
        Dense(16, activation='relu'), Dense(1),
    ])
    model.compile(optimizer='adam', loss='mse')
    es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
    model.fit(Xtr, ytr, epochs=epochs, batch_size=16, validation_split=0.15,
              callbacks=[es], verbose=verbose, shuffle=False)

    pred_s = model.predict(Xte, verbose=0).ravel()
    pred = ys.inverse_transform(pred_s.reshape(-1, 1)).ravel()

    # map back onto te_pos (all holdout rows have >=12 months of history here)
    out = np.full(len(te_pos), np.nan)
    pos_of = {p: k for k, p in enumerate(te_pos)}
    for v, i in zip(pred, kept):
        out[pos_of[i]] = v
    return out, model


def run_hybrid(X, y, tr_pos, te_pos):
    """SARIMA captures the seasonal signal; XGBoost models what SARIMA leaves behind."""
    set_seeds()
    fitted_tr, fc_te = fit_sarima(y[tr_pos], len(te_pos))
    resid_tr = y[tr_pos] - fitted_tr

    m = xgb.XGBRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                         subsample=0.8, colsample_bytree=0.8,
                         random_state=SEED, n_jobs=-1, objective='reg:squarederror')
    m.fit(X[tr_pos], resid_tr, verbose=False)
    return fc_te + m.predict(X[te_pos]), m


MODELS = {
    'Random Forest': run_rf,
    'XGBoost'      : run_xgb,
    'SVR'          : run_svr,
    'LSTM'         : run_lstm,
    'Hybrid'       : run_hybrid,
}
print('Models registered:', ', '.join(MODELS))

Models registered: Random Forest, XGBoost, SVR, LSTM, Hybrid


## 8. Holdout run (2023-01 to 2024-12)

Each model checkpoints to Drive as it finishes, so a Colab disconnect during LSTM
training does not cost the tree-based results.

In [31]:
holdout_pred = {}
holdout_rows = []

for name, fn in MODELS.items():
    print(f'--- {name} ---')
    pred, _ = fn(X_all, y_all, tr_idx, te_idx)
    holdout_pred[name] = pred
    m = metrics(y_all[te_idx], pred, name)
    holdout_rows.append(m)
    print(f"  RMSE {m['RMSE']:.4f} | MAE {m['MAE']:.4f} | "
          f"MAPE {m['MAPE']:.2f}% | R2 {m['R2']:.4f}")
    save_ckpt(f'holdout_{name.replace(" ", "_")}',
              {'metrics': m, 'pred': [float(v) for v in pred]})

# Add R's canonical SARIMA row
sar_m = metrics(r_sarima['Actual'].to_numpy(),
                r_sarima['SARIMA_Forecast'].to_numpy(), 'SARIMA')
holdout_rows.append(sar_m)

holdout_tbl = (pd.DataFrame(holdout_rows)
                 .sort_values('RMSE')
                 .reset_index(drop=True))
holdout_tbl.insert(0, 'Rank', holdout_tbl.index + 1)
holdout_tbl.to_csv(os.path.join(OUT_DATA, '01_holdout_metrics_all_models.csv'), index=False)
print()
print(holdout_tbl.to_string(index=False))

--- Random Forest ---
  RMSE 0.2360 | MAE 0.1679 | MAPE 6.44% | R2 0.8279
  checkpoint -> Checkpoints/holdout_Random_Forest.json
--- XGBoost ---
  RMSE 0.2575 | MAE 0.1774 | MAPE 6.70% | R2 0.7952
  checkpoint -> Checkpoints/holdout_XGBoost.json
--- SVR ---
  RMSE 0.3693 | MAE 0.2651 | MAPE 9.99% | R2 0.5788
  checkpoint -> Checkpoints/holdout_SVR.json
--- LSTM ---
  RMSE 0.2496 | MAE 0.1753 | MAPE 6.99% | R2 0.8076
  checkpoint -> Checkpoints/holdout_LSTM.json
--- Hybrid ---
  RMSE 0.2799 | MAE 0.2043 | MAPE 7.99% | R2 0.7580
  checkpoint -> Checkpoints/holdout_Hybrid.json

 Rank         Model   RMSE    MAE   MAPE     R2  n
    1 Random Forest 0.2360 0.1679 6.4388 0.8279 24
    2        SARIMA 0.2469 0.1648 6.3801 0.8116 24
    3          LSTM 0.2496 0.1753 6.9893 0.8076 24
    4       XGBoost 0.2575 0.1774 6.7017 0.7952 24
    5        Hybrid 0.2799 0.2043 7.9881 0.7580 24
    6           SVR 0.3693 0.2651 9.9897 0.5788 24


## 9. Rolling-origin cross-validation

Five folds of 24 months with an expanding training window, numbered to match the R output:
**Fold 1 = 2023-01..2024-12** (identical to the holdout, which is the consistency check) down
to **Fold 5 = 2015-01..2016-12**.

Every fold refits from scratch, including its scalers. This is the expensive part of the
notebook - roughly 25 model fits, of which the 5 LSTM fits dominate. Expect a few minutes.

In [32]:
K, W = 5, 24
n = len(mat)

folds = []
for k in range(K, 0, -1):
    te_end   = n - (K - k) * W          # exclusive
    te_start = te_end - W
    folds.append({
        'Fold': K - k + 1,
        'tr_pos': np.arange(0, te_start),
        'te_pos': np.arange(te_start, te_end),
    })

print('Fold layout')
for f in folds:
    d0 = pd.Timestamp(dates_all[f['te_pos'][0]]).strftime('%Y-%m')
    d1 = pd.Timestamp(dates_all[f['te_pos'][-1]]).strftime('%Y-%m')
    print(f"  Fold {f['Fold']}: train n={len(f['tr_pos']):3d} | test {d0} -> {d1}")

assert (pd.Timestamp(dates_all[folds[0]['te_pos'][0]]) == TEST_START), (
    'Fold 1 must equal the holdout window')

Fold layout
  Fold 1: train n=492 | test 2023-01 -> 2024-12
  Fold 2: train n=468 | test 2021-01 -> 2022-12
  Fold 3: train n=444 | test 2019-01 -> 2020-12
  Fold 4: train n=420 | test 2017-01 -> 2018-12
  Fold 5: train n=396 | test 2015-01 -> 2016-12


In [33]:
cv_rows, cv_fc_rows = [], []

for f in folds:
    fold, trp, tep = f['Fold'], f['tr_pos'], f['te_pos']
    print(f'=== Fold {fold} (train n={len(trp)}) ===')

    for name, fn in MODELS.items():
        try:
            pred, _ = fn(X_all, y_all, trp, tep)
        except Exception as ex:
            print(f'  {name}: FAILED ({ex})')
            continue

        m = metrics(y_all[tep], pred, name)
        m['Fold'] = fold
        m['Train_n'] = int(len(trp))
        m['Test_start'] = pd.Timestamp(dates_all[tep[0]]).strftime('%Y-%m')
        m['Test_end']   = pd.Timestamp(dates_all[tep[-1]]).strftime('%Y-%m')
        cv_rows.append(m)
        print(f"  {name:<14} RMSE {m['RMSE']:.4f}  R2 {m['R2']:.4f}")

        for i, p in zip(tep, pred):
            cv_fc_rows.append({
                'Fold': fold,
                'Date': pd.Timestamp(dates_all[i]).strftime('%Y-%m-%d'),
                'Model': name,
                'Actual': round(float(y_all[i]), 6),
                'Forecast': round(float(p), 6),
                'Error': round(float(y_all[i] - p), 6),
            })

    save_ckpt(f'cv_fold_{fold}', cv_rows)

cv_fold_tbl = pd.DataFrame(cv_rows)[
    ['Fold','Model','Train_n','Test_start','Test_end','RMSE','MAE','MAPE','R2']]
cv_fold_tbl.to_csv(os.path.join(OUT_DATA, '02_cv_fold_metrics_ml.csv'), index=False)
print(); print('Fold-level metrics saved.')
cv_fold_tbl.head(10)

=== Fold 1 (train n=492) ===
  Random Forest  RMSE 0.2360  R2 0.8279
  XGBoost        RMSE 0.2575  R2 0.7952
  SVR            RMSE 0.3693  R2 0.5788
  LSTM           RMSE 0.2496  R2 0.8076
  Hybrid         RMSE 0.2799  R2 0.7580
  checkpoint -> Checkpoints/cv_fold_1.json
=== Fold 2 (train n=468) ===
  Random Forest  RMSE 0.1980  R2 0.8978
  XGBoost        RMSE 0.2315  R2 0.8602
  SVR            RMSE 0.2813  R2 0.7937
  LSTM           RMSE 0.2857  R2 0.7872
  Hybrid         RMSE 0.2796  R2 0.7961
  checkpoint -> Checkpoints/cv_fold_2.json
=== Fold 3 (train n=444) ===
  Random Forest  RMSE 0.2551  R2 0.8332
  XGBoost        RMSE 0.2476  R2 0.8428
  SVR            RMSE 0.2522  R2 0.8370
  LSTM           RMSE 0.3335  R2 0.7150
  Hybrid         RMSE 0.2524  R2 0.8368
  checkpoint -> Checkpoints/cv_fold_3.json
=== Fold 4 (train n=420) ===
  Random Forest  RMSE 0.2782  R2 0.7611
  XGBoost        RMSE 0.2951  R2 0.7313
  SVR            RMSE 0.3363  R2 0.6509
  LSTM           RMSE 0.3003  R2 0.

,Fold,Model,Train_n,Test_start,Test_end,RMSE,MAE,MAPE,R2
0,1,Random Forest,492,2023-01,2024-12,0.2360,0.1679,6.4388,0.8279
1,1,XGBoost,492,2023-01,2024-12,0.2575,0.1774,6.7017,0.7952
2,1,SVR,492,2023-01,2024-12,0.3693,0.2651,9.9897,0.5788
3,1,LSTM,492,2023-01,2024-12,0.2496,0.1753,6.9893,0.8076
4,1,Hybrid,492,2023-01,2024-12,0.2799,0.2043,7.9881,0.7580
5,2,Random Forest,468,2021-01,2022-12,0.1980,0.1498,5.3765,0.8978
6,2,XGBoost,468,2021-01,2022-12,0.2315,0.1848,6.4212,0.8602
7,2,SVR,468,2021-01,2022-12,0.2813,0.2278,8.2148,0.7937
8,2,LSTM,468,2021-01,2022-12,0.2857,0.2088,7.4553,0.7872
9,2,Hybrid,468,2021-01,2022-12,0.2796,0.2052,6.8310,0.7961


In [34]:
# --- summarise across folds, and append R's canonical SARIMA fold metrics -----
r_cv = pd.read_csv(os.path.join(ML_IN, '10c_cv_forecasts_sarima.csv'), parse_dates=['Date'])

sar_rows = []
for fold, g in r_cv.groupby('Fold'):
    m = metrics(g['Actual'].to_numpy(), g['Forecast'].to_numpy(), 'SARIMA')
    m['Fold'] = int(fold)
    sar_rows.append(m)

cv_all = pd.concat([pd.DataFrame(cv_rows), pd.DataFrame(sar_rows)], ignore_index=True)

cv_summary = (cv_all.groupby('Model')
    .agg(n_folds=('RMSE', 'size'),
         RMSE_mean=('RMSE', 'mean'), RMSE_sd=('RMSE', 'std'),
         MAE_mean=('MAE', 'mean'),
         MAPE_mean=('MAPE', 'mean'),
         R2_mean=('R2', 'mean'))
    .round(4).sort_values('RMSE_mean').reset_index())
cv_summary.insert(0, 'Rank', range(1, len(cv_summary) + 1))
cv_summary.to_csv(os.path.join(OUT_DATA, '03_cv_summary_all_models.csv'), index=False)
print(cv_summary.to_string(index=False))

 Rank         Model  n_folds  RMSE_mean  RMSE_sd  MAE_mean  MAPE_mean  R2_mean
    1 Random Forest        5     0.2639   0.0575    0.2106     8.1582   0.8191
    2        SARIMA        5     0.2672   0.0416    0.1978     7.4878   0.8169
    3       XGBoost        5     0.2718   0.0389    0.2146     8.1899   0.8071
    4        Hybrid        5     0.2896   0.0387    0.2265     8.5423   0.7839
    5          LSTM        5     0.3076   0.0457    0.2359     9.2038   0.7570
    6           SVR        5     0.3341   0.0710    0.2644    10.2851   0.7048


## 10. Merge into the R forecast files

Merging on `Date` rather than position, so a misalignment surfaces as a NaN instead of
silently shifting a series. The merged files feed the Diebold-Mariano tests, which run in R
(`forecast::dm.test`) - that stays the canonical location.

In [35]:
# ---- holdout: one column per model, aligned to the R file -------------------
merged = r_sarima.copy()
hold_dates = pd.to_datetime(mat.loc[te_idx, 'Date'].to_numpy())

for name, pred in holdout_pred.items():
    col = name.replace(' ', '_')
    s = pd.Series(pred, index=hold_dates)
    merged[col] = merged['Date'].map(s).round(6)
    merged[f'{col}_Error'] = (merged['Actual'] - merged[col]).round(6)

na = int(merged.filter(regex='_Error$').isna().sum().sum())
assert na == 0, f'{na} NaN after merge - dates did not align'

merged.to_csv(os.path.join(OUT_DATA, '04_all_forecasts_merged_holdout.csv'), index=False)
print('holdout merged :', merged.shape)

# ---- CV: long format, SARIMA + every ML model ------------------------------
cv_fc = pd.DataFrame(cv_fc_rows)
sar_long = r_cv.assign(Model='SARIMA')[['Fold','Date','Model','Actual','Forecast','Error']]
sar_long['Date'] = pd.to_datetime(sar_long['Date']).dt.strftime('%Y-%m-%d')
cv_fc_all = pd.concat([sar_long, cv_fc], ignore_index=True)
cv_fc_all.to_csv(os.path.join(OUT_DATA, '05_all_forecasts_merged_cv.csv'), index=False)

print('cv merged      :', cv_fc_all.shape)
print('rows per model :')
print(cv_fc_all.Model.value_counts().to_string())

holdout merged : (24, 19)
cv merged      : (720, 6)
rows per model :
Model
SARIMA           120
Random Forest    120
XGBoost          120
SVR              120
LSTM             120
Hybrid           120


In [36]:
# ---- holdout error by year: does every model miss April 2023 the same way? --
by_year = []
tmp = merged.copy()
tmp['Year'] = tmp['Date'].dt.year
for col in ['SARIMA_Forecast'] + [n.replace(' ', '_') for n in holdout_pred]:
    name = 'SARIMA' if col == 'SARIMA_Forecast' else col.replace('_', ' ')
    for yr, g in tmp.groupby('Year'):
        e = (g['Actual'] - g[col]).to_numpy()
        by_year.append({'Model': name, 'Year': int(yr), 'n': len(e),
                        'Mean_Error': round(float(e.mean()), 5),
                        'RMSE': round(float(np.sqrt((e ** 2).mean())), 5)})

by_year = pd.DataFrame(by_year).pivot(index='Model', columns='Year',
                                      values='RMSE').reset_index()
by_year.columns = ['Model'] + [f'RMSE_{c}' for c in by_year.columns[1:]]

apr = tmp[tmp['Date'] == '2023-04-01']
apr_row = {'Model': 'SARIMA',
           'Apr2023_Error': round(float(apr['Actual'].iloc[0]
                                        - apr['SARIMA_Forecast'].iloc[0]), 5)}
apr_rows = [apr_row] + [
    {'Model': n, 'Apr2023_Error': round(float(apr['Actual'].iloc[0]
                                              - apr[n.replace(' ', '_')].iloc[0]), 5)}
    for n in holdout_pred]

by_year = by_year.merge(pd.DataFrame(apr_rows), on='Model')
by_year.to_csv(os.path.join(OUT_DATA, '06_holdout_error_by_year.csv'), index=False)
print(by_year.to_string(index=False))
print()
print('A large negative Apr2023 error across ALL models points to a data anomaly')
print('rather than a model failure - worth one sentence in the discussion.')

        Model  RMSE_2023  RMSE_2024  Apr2023_Error
       Hybrid    0.37446    0.12817       -0.90952
         LSTM    0.31243    0.16414       -0.69273
Random Forest    0.29880    0.14877       -0.79726
       SARIMA    0.32436    0.12936       -0.81108
          SVR    0.45696    0.25280       -0.89089
      XGBoost    0.29950    0.20717       -0.79851

A large negative Apr2023 error across ALL models points to a data anomaly
rather than a model failure - worth one sentence in the discussion.


## 11. Figures

In [37]:
plt.rcParams.update({'figure.dpi': 150, 'savefig.dpi': 300,
                     'font.size': 9, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.axisbelow': True})

order = holdout_tbl['Model'].tolist()
colors = plt.cm.tab10(np.linspace(0, 1, 10))

# --- Fig1: holdout RMSE + CV RMSE side by side ------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

h = holdout_tbl.sort_values('RMSE')
b = ax[0].bar(h['Model'], h['RMSE'],
              color=['#c0392b' if m == 'SARIMA' else '#7f8c8d' for m in h['Model']])
ax[0].set_ylabel('RMSE (m/s)'); ax[0].set_title('(a) Holdout RMSE, 2023-2024')
ax[0].tick_params(axis='x', rotation=35)
for r, v in zip(b, h['RMSE']):
    ax[0].text(r.get_x() + r.get_width()/2, v, f'{v:.4f}', ha='center',
               va='bottom', fontsize=7)

c = cv_summary.sort_values('RMSE_mean')
ax[1].bar(c['Model'], c['RMSE_mean'], yerr=c['RMSE_sd'], capsize=3,
          color=['#c0392b' if m == 'SARIMA' else '#7f8c8d' for m in c['Model']])
ax[1].set_ylabel('Mean CV RMSE (m/s)')
ax[1].set_title('(b) Rolling-origin CV RMSE (mean +/- sd)')
ax[1].tick_params(axis='x', rotation=35)

plt.tight_layout()
plt.savefig(os.path.join(OUT_GRAPH, 'Fig1_model_comparison_rmse.png'), bbox_inches='tight')
plt.close()
print('[png] Fig1_model_comparison_rmse.png')

[png] Fig1_model_comparison_rmse.png


In [38]:
# --- Fig2: holdout forecasts vs actual --------------------------------------
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(merged['Date'], merged['Actual'], 'k-o', lw=2, ms=4, label='Actual', zorder=10)
ax.plot(merged['Date'], merged['SARIMA_Forecast'], '--', lw=1.8,
        color='#c0392b', label='SARIMA')
for i, name in enumerate(holdout_pred):
    ax.plot(merged['Date'], merged[name.replace(' ', '_')], '-', lw=1.1,
            alpha=0.8, color=colors[i], label=name)
ax.set_ylabel('Wind speed (m/s)'); ax.set_xlabel('')
ax.set_title('Holdout forecasts, Jan 2023 - Dec 2024')
ax.legend(ncol=3, fontsize=8, frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_GRAPH, 'Fig2_holdout_forecasts.png'), bbox_inches='tight')
plt.close()
print('[png] Fig2_holdout_forecasts.png')

# --- Fig3: error distributions ----------------------------------------------
fig, ax = plt.subplots(figsize=(9, 4.2))
err_cols = ['SARIMA'] + list(holdout_pred)
data = ([merged['Error'].to_numpy()]
        + [merged[f"{n.replace(' ', '_')}_Error"].to_numpy() for n in holdout_pred])
bp = ax.boxplot(data, labels=err_cols, patch_artist=True, showmeans=True)
for patch, m in zip(bp['boxes'], err_cols):
    patch.set_facecolor('#c0392b' if m == 'SARIMA' else '#bdc3c7')
    patch.set_alpha(0.7)
ax.axhline(0, color='red', ls='--', lw=1)
ax.set_ylabel('Forecast error (m/s)')
ax.set_title('Holdout error distribution by model')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig(os.path.join(OUT_GRAPH, 'Fig3_error_distributions.png'), bbox_inches='tight')
plt.close()
print('[png] Fig3_error_distributions.png')

[png] Fig2_holdout_forecasts.png
[png] Fig3_error_distributions.png


In [39]:
# --- Fig4: feature importance (RF + XGBoost) --------------------------------
set_seeds()
rf_full = RandomForestRegressor(n_estimators=500, max_depth=12, min_samples_leaf=2,
                                max_features='sqrt', random_state=SEED, n_jobs=-1)
rf_full.fit(X_all[tr_idx], y_all[tr_idx])

xgb_full = xgb.XGBRegressor(n_estimators=500, max_depth=4, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                            random_state=SEED, n_jobs=-1,
                            objective='reg:squarederror')
xgb_full.fit(X_all[tr_idx], y_all[tr_idx], verbose=False)

imp = pd.DataFrame({'Feature': FEATURES,
                    'RF': rf_full.feature_importances_,
                    'XGBoost': xgb_full.feature_importances_}).round(5)
imp['Mean'] = imp[['RF', 'XGBoost']].mean(axis=1).round(5)
imp = imp.sort_values('Mean', ascending=False).reset_index(drop=True)
imp.to_csv(os.path.join(OUT_DATA, '07_feature_importance.csv'), index=False)

top = imp.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5.5))
yy = np.arange(len(top))
ax.barh(yy - 0.2, top['RF'], height=0.4, label='Random Forest', color='#2980b9')
ax.barh(yy + 0.2, top['XGBoost'], height=0.4, label='XGBoost', color='#e67e22')
ax.set_yticks(yy); ax.set_yticklabels(top['Feature'])
ax.set_xlabel('Relative importance')
ax.set_title('Top 15 features (fitted on training window only)')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(OUT_GRAPH, 'Fig4_feature_importance.png'), bbox_inches='tight')
plt.close()
print('[png] Fig4_feature_importance.png'); print()
print(imp.head(10).to_string(index=False))

[png] Fig4_feature_importance.png

  Feature      RF  XGBoost    Mean
 WS_lag12 0.22982  0.46405 0.34693
       PS 0.14143  0.13352 0.13748
    Month 0.09016  0.15009 0.12012
      T2M 0.14236  0.06711 0.10474
  WS_lag6 0.10014  0.01562 0.05788
  WS_lag1 0.07023  0.01197 0.04110
 WS_roll3 0.02950  0.01901 0.02425
     RH2M 0.02157  0.01683 0.01920
     WD2M 0.02568  0.00951 0.01760
WS_roll12 0.01808  0.01686 0.01747


## 12. Summary

The headline test: does any ML or DL model beat SARIMA once leakage is removed?

Note that non-significance in a later DM test would reflect the 24-observation holdout's low
power, not model equivalence - the 120-observation CV comparison carries more weight.

In [40]:
best_ml = holdout_tbl[holdout_tbl.Model != 'SARIMA'].iloc[0]
sar_row = holdout_tbl[holdout_tbl.Model == 'SARIMA'].iloc[0]
cv_best_ml = cv_summary[cv_summary.Model != 'SARIMA'].iloc[0]
cv_sar = cv_summary[cv_summary.Model == 'SARIMA'].iloc[0]

summary = pd.DataFrame({
    'Item': [
        'Feature matrix', 'Train window', 'Holdout window', 'CV design',
        'SARIMA holdout RMSE', 'Best ML holdout', 'Best ML holdout RMSE',
        'SARIMA beaten on holdout?',
        'SARIMA CV mean RMSE', 'Best ML CV', 'Best ML CV mean RMSE',
        'SARIMA beaten on CV?',
        'Python/R SARIMA agreement',
    ],
    'Value': [
        f'{len(mat)} x {len(FEATURES)}',
        f'{len(tr_idx)} months (1982-01 to 2022-12)',
        f'{len(te_idx)} months (2023-01 to 2024-12)',
        f'{K} folds x {W} months, expanding',
        f"{sar_row['RMSE']:.4f}",
        best_ml['Model'], f"{best_ml['RMSE']:.4f}",
        'YES' if best_ml['RMSE'] < sar_row['RMSE'] else 'NO',
        f"{cv_sar['RMSE_mean']:.4f}",
        cv_best_ml['Model'], f"{cv_best_ml['RMSE_mean']:.4f}",
        'YES' if cv_best_ml['RMSE_mean'] < cv_sar['RMSE_mean'] else 'NO',
        f"|diff| = {abs(py_sarima['RMSE'] - r_rmse):.4f}",
    ]})
summary.to_csv(os.path.join(OUT_DATA, '08_SUMMARY_ml.csv'), index=False)
print(summary.to_string(index=False))

print(); print('=' * 62)
print('OUTPUTS')
print('=' * 62)
for lbl, path in [('Data', OUT_DATA), ('Graph', OUT_GRAPH)]:
    print(f'{lbl}:')
    for f in sorted(os.listdir(path)):
        print('   ', f)

print(); print('Next: run the Diebold-Mariano tests in R using')
print('  ML_Output/Data/04_all_forecasts_merged_holdout.csv')
print('  ML_Output/Data/05_all_forecasts_merged_cv.csv')

                     Item                           Value
           Feature matrix                        516 x 21
             Train window 492 months (1982-01 to 2022-12)
           Holdout window  24 months (2023-01 to 2024-12)
                CV design  5 folds x 24 months, expanding
      SARIMA holdout RMSE                          0.2469
          Best ML holdout                   Random Forest
     Best ML holdout RMSE                          0.2360
SARIMA beaten on holdout?                             YES
      SARIMA CV mean RMSE                          0.2672
               Best ML CV                   Random Forest
     Best ML CV mean RMSE                          0.2639
     SARIMA beaten on CV?                             YES
Python/R SARIMA agreement                 |diff| = 0.0005

OUTPUTS
Data:
    01_holdout_metrics_all_models.csv
    02_cv_fold_metrics_ml.csv
    03_cv_summary_all_models.csv
    04_all_forecasts_merged_holdout.csv
    05_all_forecasts_merged_cv.c